In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

# --- 1. The Metric: Maximum Mean Discrepancy (Gaussian Kernel) ---
def guassian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    n_samples = int(source.size()[0]) + int(target.size()[0])
    total = torch.cat([source, target], dim=0)

    total0 = total.unsqueeze(0).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    total1 = total.unsqueeze(1).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    L2_distance = ((total0 - total1)**2).sum(2)

    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(L2_distance.data) / (n_samples**2 - n_samples)

    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul**i) for i in range(kernel_num)]

    kernel_val = [torch.exp(-L2_distance / bandwidth_temp) for bandwidth_temp in bandwidth_list]
    return sum(kernel_val)

def compute_mmd(source, target):
    """
    Computes MMD between two sets of model outputs (N_models x Output_Dim).
    """
    batch_size = int(source.size()[0])
    kernels = guassian_kernel(source, target)
    XX = kernels[:batch_size, :batch_size]
    YY = kernels[batch_size:, batch_size:]
    XY = kernels[:batch_size, batch_size:]
    loss = torch.mean(XX + YY - 2 * XY)
    return loss

In [ ]:
# --- 2. Setup Data & Models ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_resnet_tiny():
    # Using ResNet18 for speed, modified for CIFAR size
    model = torchvision.models.resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)

# Data Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Full CIFAR
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# SPLIT: Train A (Small for Overfit), Train B (Large for General), Holdout
indices = list(range(len(trainset)))
# Group O Data: First 2000 images (Easy to memorize)
indices_O = indices[:2000]
# Group G Data: Next 40000 images
indices_G = indices[2000:42000]
# Holdout (Unlabeled for test): Last 8000 images
indices_Holdout = indices[42000:]

loader_O = DataLoader(Subset(trainset, indices_O), batch_size=128, shuffle=True)
loader_G = DataLoader(Subset(trainset, indices_G), batch_size=128, shuffle=True)
loader_Holdout = DataLoader(Subset(trainset, indices_Holdout), batch_size=128, shuffle=False)

# --- 3. Training Function ---
def train_model(mode="general"):
    model = get_resnet_tiny()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()

    loader = loader_G if mode == "general" else loader_O
    epochs = 10 if mode == "general" else 30 # Overfit needs more epochs on small data to zero out

    model.train()
    print(f"Training Model ({mode})...")
    for epoch in range(epochs):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # CRITICAL: For Group O (Random Labels variant), scramble targets
            # Uncomment below to test "Random Label" overfitting specifically
            # if mode == "overfit":
            #     targets = torch.randint(0, 10, targets.shape).to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
    return model

# --- 4. Feature Extraction on Holdout ---
def get_holdout_representations(model):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for inputs, _ in loader_Holdout:
            inputs = inputs.to(device)
            preds = model(inputs) # Logits
            all_preds.append(preds)
    return torch.cat(all_preds).cpu()

# --- 5. Execution ---

# Train Group G (3 models for split, usually 5 is better)
print("Generating Group G (General)...")
models_G = [train_model("general") for _ in range(4)]
rep_G = [get_holdout_representations(m) for m in models_G]

# Train Group O (2 models)
print("Generating Group O (Overfit)...")
models_O = [train_model("overfit") for _ in range(2)]
rep_O = [get_holdout_representations(m) for m in models_O]

# Flatten representations for MMD: (N_models, N_samples * N_classes)
# We treat the entire holdout prediction vector of a model as ONE sample point in "Model Space"
flat_G = torch.stack([r.flatten() for r in rep_G])
flat_O = torch.stack([r.flatten() for r in rep_O])

# Split G for D_self
half_G = len(flat_G) // 2
flat_G1 = flat_G[:half_G]
flat_G2 = flat_G[half_G:]

# --- 6. Calculate Distances ---
d_self = compute_mmd(flat_G1, flat_G2).item()
d_cross = compute_mmd(flat_O, flat_G).item()

print("-" * 30)
print(f"D_self (Control): {d_self:.6f}")
print(f"D_cross (Test):   {d_cross:.6f}")

if d_cross > (d_self * 2): # Heuristic threshold
    print("RESULT: SUCCESS. Overfitting is a distinct topological region.")
else:
    print("RESULT: FAILURE. Overfitting is indistinguishable from Generalization.")

------------------------------
D_self (Control): 2.926587
D_cross (Test):   3.843546
RESULT: FAILURE. Overfitting is indistinguishable from Generalization.


In [ ]:
def compute_mmd(source, target):
    """
    Computes MMD between two sets of model outputs with potentially different batch sizes.
    """
    n_source = int(source.size()[0])
    n_target = int(target.size()[0])

    # 1. Compute the full Kernel Matrix (Size: (n_source+n_target) x (n_source+n_target))
    kernels = guassian_kernel(source, target)

    # 2. Slice the kernel matrix into blocks
    # XX: Similarity within Source Group (n_source x n_source)
    XX = kernels[:n_source, :n_source]

    # YY: Similarity within Target Group (n_target x n_target)
    YY = kernels[n_source:, n_source:]

    # XY: Similarity between Source and Target (n_source x n_target)
    XY = kernels[:n_source, n_source:]

    # 3. Compute MMD = Mean(XX) + Mean(YY) - 2 * Mean(XY)
    # We compute the mean of each block independently to handle size mismatches
    loss = torch.mean(XX) + torch.mean(YY) - 2 * torch.mean(XY)

    return loss

In [ ]:
# --- Re-Test with Varying Kernel Bandwidths ---

def compute_mmd_with_scale(source, target, scale_factor=1.0):
    n_source = int(source.size()[0])
    n_target = int(target.size()[0])
    total = torch.cat([source, target], dim=0)

    # 1. Calculate L2 Distance Matrix
    total0 = total.unsqueeze(0).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    total1 = total.unsqueeze(1).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    L2_distance = ((total0 - total1)**2).sum(2)

    # 2. Determine Base Sigma (Heuristic)
    n_samples = n_source + n_target
    base_sigma = torch.sum(L2_distance.data) / (n_samples**2 - n_samples)

    # 3. Apply Scaling ("Zoom")
    # scale_factor < 1.0 means "Sharper" (Zoom In)
    fix_sigma = base_sigma * scale_factor

    # 4. Compute Kernel using Fixed Sigma
    # We use a single kernel here for clarity, or a tight range around the fixed sigma
    kernel_val = torch.exp(-L2_distance / fix_sigma)

    # 5. MMD Logic (Block Mean)
    XX = kernel_val[:n_source, :n_source]
    YY = kernel_val[n_source:, n_source:]
    XY = kernel_val[:n_source, n_source:]

    return torch.mean(XX) + torch.mean(YY) - 2 * torch.mean(XY)

print(f"{'Zoom Level':<12} | {'Scale':<6} | {'D_self':<10} | {'D_cross':<10} | {'Ratio (Target > 2.0)':<20}")
print("-" * 65)

# We test scales from 1.0 (Original) down to 0.05 (Extreme Detail)
scales = [1.0, 0.5, 0.25, 0.1, 0.05]

for s in scales:
    d_s = compute_mmd_with_scale(flat_G1, flat_G2, scale_factor=s).item()
    d_c = compute_mmd_with_scale(flat_O, flat_G, scale_factor=s).item()

    ratio = d_c / (d_s + 1e-9) # Avoid div by zero

    zoom_desc = "Normal"
    if s <= 0.5: zoom_desc = "High"
    if s <= 0.1: zoom_desc = "Extreme"

    print(f"{zoom_desc:<12} | {s:<6.2f} | {d_s:<10.4f} | {d_c:<10.4f} | {ratio:<20.2f}")

Zoom Level   | Scale  | D_self     | D_cross    | Ratio (Target > 2.0)
-----------------------------------------------------------------
Normal       | 1.00   | 0.5841     | 0.8955     | 1.53                
High         | 0.50   | 0.8203     | 1.0143     | 1.24                
High         | 0.25   | 0.9635     | 0.9231     | 0.96                
Extreme      | 0.10   | 0.9996     | 0.7747     | 0.77                
Extreme      | 0.05   | 1.0000     | 0.7512     | 0.75                


In [ ]:
import torch.nn.functional as F

def compute_local_separation(rep_G, rep_O):
    """
    Computes separation per-image, then averages.

    Args:
        rep_G: List of tensors, each (N_samples, 10)
        rep_O: List of tensors, each (N_samples, 10)
    """
    # Stack to shape: (N_models, N_samples, 10)
    stack_G = torch.stack(rep_G) # [4, 8000, 10]
    stack_O = torch.stack(rep_O) # [2, 8000, 10]

    # Permute to: (N_samples, N_models, 10)
    # Now we iterate over samples (images)
    data_G = stack_G.permute(1, 0, 2)
    data_O = stack_O.permute(1, 0, 2)

    n_samples = data_G.size(0)

    total_d_self = 0.0
    total_d_cross = 0.0

    # We batch this to avoid loop slowness, but conceptually it is per-image
    # Reshape for MMD: We treat the (N_models) predictions as the 'distribution' for that image.

    print(f"Analyzing local geometry across {n_samples} images...")

    # Simple Euclidean Centroid Distance (Faster & often more robust than MMD for small point counts)
    # 1. Calculate Centroid of Group G for each image
    centroid_G = torch.mean(data_G, dim=1) # [8000, 10]

    # 2. D_self: Average distance of G members to their own centroid
    # (How spread out is the General Cluster?)
    diff_self = data_G - centroid_G.unsqueeze(1) # [8000, 4, 10]
    dist_self = torch.norm(diff_self, dim=2).mean(dim=1) # [8000]

    # 3. D_cross: Average distance of O members to Group G Centroid
    # (How far is the Overfit Cluster from the General Center?)
    diff_cross = data_O - centroid_G.unsqueeze(1) # [8000, 2, 10]
    dist_cross = torch.norm(diff_cross, dim=2).mean(dim=1) # [8000]

    # Global Averages
    avg_d_self = dist_self.mean().item()
    avg_d_cross = dist_cross.mean().item()

    return avg_d_self, avg_d_cross

# --- Execute ---
# Ensure inputs are logits (not probabilities) for better geometric properties
# If your previous rep_G were probabilities, this is still fine, but logits are better.

d_self_local, d_cross_local = compute_local_separation(rep_G, rep_O)

print("-" * 40)
print("PER-IMAGE GEOMETRY TEST")
print("-" * 40)
print(f"Avg Radius of General Cluster (D_self): {d_self_local:.4f}")
print(f"Avg Distance to Overfit Models (D_cross): {d_cross_local:.4f}")

ratio = d_cross_local / (d_self_local + 1e-9)
print(f"Ratio: {ratio:.4f}")

if ratio > 1.5:
    print("RESULT: SUCCESS. Overfit models are systematically offset from the General consensus.")
else:
    print("RESULT: FAILURE. Overfit models hide inside the variance of General models.")

Analyzing local geometry across 8000 images...
----------------------------------------
PER-IMAGE GEOMETRY TEST
----------------------------------------
Avg Radius of General Cluster (D_self): 4.0128
Avg Distance to Overfit Models (D_cross): 12.3375
Ratio: 3.0745
RESULT: SUCCESS. Overfit models are systematically offset from the General consensus.


In [ ]:
def check_behavior(model_group, name):
    correct = 0
    total = 0
    norms = []

    with torch.no_grad():
        for inputs, targets in loader_Holdout:
            inputs, targets = inputs.to(device), targets.to(device)

            # Average predictions across the ensemble
            outputs_list = [m(inputs) for m in model_group]
            avg_output = torch.stack(outputs_list).mean(0)

            # Check Accuracy
            _, predicted = torch.max(avg_output.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

            # Check Norm (Magnitude of the logit vector)
            # High norm = High confidence/sharpness
            norms.append(torch.norm(avg_output, dim=1))

    avg_norm = torch.cat(norms).mean().item()
    acc = 100 * correct / total
    print(f"Group {name} | Accuracy: {acc:.2f}% | Avg Logit Norm: {avg_norm:.4f}")

print("-" * 50)
check_behavior(models_G, "G (General)")
check_behavior(models_O, "O (Overfit)")
print("-" * 50)

--------------------------------------------------
Group G (General) | Accuracy: 85.55% | Avg Logit Norm: 9.9726
Group O (Overfit) | Accuracy: 48.21% | Avg Logit Norm: 14.4136
--------------------------------------------------


In [ ]:
# --- 1. Generate Group N (Null / Untrained) ---
print("Generating Group N (Null/Random)...")
models_N = [get_resnet_tiny() for _ in range(2)] # Just init, no train loop
rep_N = [get_holdout_representations(m) for m in models_N]

# --- 2. Calculate All Pairwise Distances ---

# We already have:
# d_self_local (G vs G) -> The "Cluster Size" of Good Models
# d_cross_local (G vs O) -> The Distance to the "Overfit Trap"

# We need:
# D_null (G vs N) -> Distance to "Randomness"
d_null_self, d_null = compute_local_separation(rep_G, rep_N)

# D_O_N (O vs N) -> Distance between "Overfit" and "Random"
# (To see if Overfitting is just Randomness in disguise)
_, d_overfit_null = compute_local_separation(rep_O, rep_N)

print("-" * 50)
print(f"{'Metric':<25} | {'Value':<10} | {'Interpretation'}")
print("-" * 50)
print(f"{'D(G, G) [Self]':<25} | {d_self_local:<10.4f} | Size of the Solution Space")
print(f"{'D(G, O) [Overfit]':<25} | {d_cross_local:<10.4f} | Distance to Overfitting")
print(f"{'D(G, N) [Random]':<25} | {d_null:<10.4f} | Distance to Random Initialization")
print(f"{'D(O, N) [Distinct?]':<25} | {d_overfit_null:<10.4f} | Is Overfitting just Randomness?")
print("-" * 50)

# --- 3. Interpretation Logic ---
if d_overfit_null < d_self_local:
    print("CONCLUSION: FAILED. Overfitting is indistinguishable from Random Noise.")
elif d_cross_local > d_null:
    print("CONCLUSION: INTERESTING. Overfitting is 'worse' (further away) than random guessing.")
else:
    print("CONCLUSION: SUCCESS. Overfitting is a distinct third state (different from Good, different from Random).")

Generating Group N (Null/Random)...
Analyzing local geometry across 8000 images...
Analyzing local geometry across 8000 images...
--------------------------------------------------
Metric                    | Value      | Interpretation
--------------------------------------------------
D(G, G) [Self]            | 4.0128     | Size of the Solution Space
D(G, O) [Overfit]         | 12.3375    | Distance to Overfitting
D(G, N) [Random]          | 10.0046    | Distance to Random Initialization
D(O, N) [Distinct?]       | 14.5541    | Is Overfitting just Randomness?
--------------------------------------------------
CONCLUSION: INTERESTING. Overfitting is 'worse' (further away) than random guessing.


In [ ]:
# --- 1. Generate Group N2 (Second Null Group) ---
print("Generating Group N2 (Null/Random Control)...")
models_N2 = [get_resnet_tiny() for _ in range(2)] # Fresh initialization
rep_N2 = [get_holdout_representations(m) for m in models_N2]

# --- 2. Calculate The Missing Links ---

# D(N1, N2) -> How consistent is "randomness"?
d_null_self, _ = compute_local_separation(rep_N, rep_N2) # rep_N is your N1

# D(N1, O) -> Distance from Null 1 to Overfit
_, d_n1_o = compute_local_separation(rep_N, rep_O)

# D(N2, O) -> Distance from Null 2 to Overfit (Should be same as N1 -> O)
_, d_n2_o = compute_local_separation(rep_N2, rep_O)

# Recall D(G, G) from previous run
d_g_self = d_self_local

print("-" * 60)
print(f"{'Metric':<30} | {'Value':<10} | {'Interpretation'}")
print("-" * 60)
print(f"{'D(G, G)   [Solution Width]':<30} | {d_g_self:<10.4f} | Coherence of Good Models")
print(f"{'D(N1, N2) [Void Width]':<30} | {d_null_self:<10.4f} | Coherence of Random Models")
print("-" * 60)
print(f"{'D(N1, O)  [Void -> Trap]':<30} | {d_n1_o:<10.4f} | Distance: Random to Overfit")
print(f"{'D(N2, O)  [Void -> Trap]':<30} | {d_n2_o:<10.4f} | Distance: Random 2 to Overfit")
print("-" * 60)

# --- 3. The Geometric Map ---
# Logic to determine where "Overfit" sits in relation to "Random" and "Good"

if d_null_self < d_g_self:
    print("INSIGHT: 'Random' is a tighter cluster than 'Trained'.")
    print("         (Untrained models all map to the origin/zero).")
else:
    print("INSIGHT: 'Random' is a dispersed cloud.")

if d_n1_o > d_null_self * 2:
    print("RESULT:  Overfitting is a DISTINCT destination.")
    print("         It is not just noise; it is a specific, learned 'Anti-Solution'.")

Generating Group N2 (Null/Random Control)...
Analyzing local geometry across 8000 images...
Analyzing local geometry across 8000 images...
Analyzing local geometry across 8000 images...
------------------------------------------------------------
Metric                         | Value      | Interpretation
------------------------------------------------------------
D(G, G)   [Solution Width]     | 4.0128     | Coherence of Good Models
D(N1, N2) [Void Width]         | 0.9817     | Coherence of Random Models
------------------------------------------------------------
D(N1, O)  [Void -> Trap]       | 15.9565    | Distance: Random to Overfit
D(N2, O)  [Void -> Trap]       | 15.9385    | Distance: Random 2 to Overfit
------------------------------------------------------------
INSIGHT: 'Random' is a tighter cluster than 'Trained'.
         (Untrained models all map to the origin/zero).
RESULT:  Overfitting is a DISTINCT destination.
         It is not just noise; it is a specific, learned